# Fourier Analysis of Real Music
### Complete Project Notebook


## Objective
- Understand Fourier Series, FT, DFT, FFT
- Decompose real audio
- Reconstruct and compute error
- Filter frequencies
- Build Streamlit UI


## Theory: Continuous Fourier Series to DFT

For a continuous periodic signal with period $T_0$, the Continuous Fourier Series (CFS) is:
$$x(t)=um_{m=-nfty}^{nfty} c_m e^{j2i m t/T_0}$$
where
$$c_m = rac{1}{T_0}nt_0^{T_0} x(t)e^{-j2i m t/T_0}dt$$

If we sample this signal at $t_n=nT_s$ and take one period with $N$ samples ($NT_s=T_0$):
$$x[n]=x(nT_s),uad n=0,1,ots,N-1$$

Approximating the CFS integral with a Riemann sum gives discrete coefficients at harmonics:
$$c_k pprox rac{1}{N}um_{n=0}^{N-1}x[n]e^{-j2i kn/N} = rac{X[k]}{N}$$
where
$$X[k]=um_{n=0}^{N-1}x[n]e^{-j2i kn/N}$$
is the DFT.

So, the DFT is the sampled-frequency version of Fourier analysis for sampled data, and it recovers discrete spectral bins:
$$f_k = rac{k}{N}f_s$$

Practical interpretation:
- Large $|X[k]|$ means strong presence of frequency $f_k$
- Phase $ngle X[k]$ controls time-domain alignment of that sinusoidal component

In [ ]:
%pip install numpy scipy matplotlib librosa soundfile streamlit

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy.fft import fft, ifft, fftfreq
import soundfile as sf
from IPython.display import Audio, display


## Load Audio

In [ ]:

file_path = "sample_audio.wav"  # change this
signal, sr = librosa.load(file_path, sr=None, mono=True)
signal = signal.astype(np.float64)

print("Sampling Rate:", sr)
print("Samples:", len(signal))
print("Duration (s):", round(len(signal)/sr, 2))

plt.figure(figsize=(10,4))
plt.plot(signal)
plt.title("Time Domain Signal")
plt.xlabel("Sample Index")
plt.ylabel("Amplitude")
plt.show()

display(Audio(signal, rate=sr))


## FFT

In [ ]:

N = len(signal)
fft_vals = fft(signal)
freqs = fftfreq(N, 1/sr)
magnitude = np.abs(fft_vals) / N
phase = np.angle(fft_vals)
pos_mask = freqs >= 0
pos_freqs = freqs[pos_mask]
pos_mag = magnitude[pos_mask]

plt.figure(figsize=(10,4))
plt.plot(pos_freqs, pos_mag)
plt.title("Magnitude Spectrum (Positive Frequencies)")
plt.xlabel("Hz")
plt.ylabel("Magnitude")
plt.xlim(0, min(8000, sr//2))
plt.show()


## Dominant Frequencies

In [ ]:

top_k = 10
top = np.argsort(pos_mag)[-top_k:][::-1]
dominant_freqs = pos_freqs[top]
dominant_mags = pos_mag[top]

print("Top frequency components:")
for f, m in zip(dominant_freqs, dominant_mags):
    print(f"{f:8.2f} Hz  | magnitude={m:.6f}")

print("\nInterpretation:")
print("Higher peaks correspond to stronger sinusoidal content in the audio.")
print("Clusters of peaks indicate harmonics/timbre rather than a single pure tone.")


## Reconstruction

In [ ]:

# Reconstruct using all extracted FFT components
reconstructed = ifft(fft_vals).real
mse = np.mean((signal - reconstructed)**2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(signal - reconstructed))

print("Reconstruction MSE :", mse)
print("Reconstruction RMSE:", rmse)
print("Reconstruction MAE :", mae)

plt.figure(figsize=(10,4))
plt.plot(signal[:1000], label="Original")
plt.plot(reconstructed[:1000], '--', label="Reconstructed")
plt.title("Original vs Reconstructed (first 1000 samples)")
plt.legend()
plt.show()


## Filtering

In [ ]:

def bandstop_filter(fft_values, freqs_hz, low_hz, high_hz):
    filtered = fft_values.copy()
    mask = (np.abs(freqs_hz) >= low_hz) & (np.abs(freqs_hz) <= high_hz)
    filtered[mask] = 0
    return filtered

filtered_fft = bandstop_filter(fft_vals, freqs, 1000, 3000)
filtered_signal = ifft(filtered_fft).real.astype(np.float32)

sf.write("filtered.wav", filtered_signal, sr)
print("Saved filtered.wav")
display(Audio(filtered_signal, rate=sr))


## Component Visualization

In [ ]:

def component_from_bin(fft_values, idx):
    comp_fft = np.zeros_like(fft_values)
    comp_fft[idx] = fft_values[idx]
    if idx != 0 and idx != len(fft_values)//2:
        comp_fft[-idx] = fft_values[-idx]
    return ifft(comp_fft).real

plt.figure(figsize=(10,4))
components = []
for i in top[:5]:
    freq = int(np.where(freqs == pos_freqs[i])[0][0])
    comp = component_from_bin(fft_vals, freq)
    components.append(comp)
    plt.plot(comp[:500], label=f"{pos_freqs[i]:.1f} Hz")

# Compare partial reconstruction from top components
partial_recon = np.sum(components, axis=0)
partial_error = np.mean((signal - partial_recon)**2)
print("Partial reconstruction MSE (top 5 components):", partial_error)

plt.legend()
plt.title("Time-domain Components (Top 5 Frequency Bins)")
plt.show()


## Streamlit App Code

In [ ]:

# Save as app.py

import streamlit as st
import numpy as np
import librosa
from scipy.fft import fft, ifft, fftfreq
import soundfile as sf
from tempfile import NamedTemporaryFile

st.title("Fourier Audio Analyzer")
st.write("Remove frequencies, isolate frequencies, or generate a tone to play any required frequency.")

file = st.file_uploader("Upload Audio", type=["wav","mp3","aac"])

if file:
    signal, sr = librosa.load(file, sr=None)
    signal = signal.astype(np.float64)

    fft_vals = fft(signal)
    freqs = fftfreq(len(signal), 1/sr)
    mag = np.abs(fft_vals) / len(signal)

    st.line_chart(mag[:len(mag)//2], height=250)

    st.subheader("Frequency Operation")
    mode = st.radio("Mode", ["Remove Band (Band-stop)", "Keep Only Band (Band-pass)"])
    nyq = int(sr // 2)
    low = st.slider("Low (Hz)", 0, nyq, min(500, nyq))
    high = st.slider("High (Hz)", 0, nyq, min(3000, nyq))

    if low > high:
        low, high = high, low

    if st.button("Apply Operation"):
        filt = fft_vals.copy()
        band_mask = (np.abs(freqs) >= low) & (np.abs(freqs) <= high)

        if mode == "Remove Band (Band-stop)":
            filt[band_mask] = 0
        else:
            filt[~band_mask] = 0

        out = ifft(filt).real
        with NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
            sf.write(tmp.name, out, sr)
            st.audio(tmp.name)
            st.success(f"Processed audio ready: {mode}, {low}-{high} Hz")

    st.subheader("Play Any Required Frequency")
    tone_freq = st.slider("Tone Frequency (Hz)", 20, nyq, 440)
    tone_duration = st.slider("Tone Duration (seconds)", 1, 5, 2)
    tone_amp = st.slider("Tone Amplitude", 0.0, 1.0, 0.3)

    if st.button("Generate Tone"):
        t = np.linspace(0, tone_duration, int(sr * tone_duration), endpoint=False)
        tone = tone_amp * np.sin(2 * np.pi * tone_freq * t)
        with NamedTemporaryFile(delete=False, suffix=".wav") as tmp_tone:
            sf.write(tmp_tone.name, tone, sr)
            st.audio(tmp_tone.name)
            st.success(f"Playing {tone_freq} Hz tone")
